# 새 전처리 파이프라인 × 모델 3개 × 시드 3개 — Colab 학습

**무엇을 하나:** 실제 현장에서 쓸 판정 순서와 똑같이 전처리한 사진으로 세 모델을 학습하고 비교한다.

1. **원본 사진(1600×1200)**에서 고정 위치를 자른다: 세로 0~1000, 가로 250~1250. 원욱님 STFPM 코드와 같은 위치다.
2. **게이트:** 자른 영역에서 밝은 부분(필라멘트)이 25% 미만이면 "판정 보류"로 빼고 학습·평가에 쓰지 않는다. 원욱님 코드의 게이트와 같은 계산이다.
3. 512×512로 줄여 모델에 넣는다. 마스킹은 하지 않는다.
4. 모델 3개(ResNet18, EfficientNet-B0, ConvNeXt-Tiny)를 `finetune` 방식으로 학습한다. 시드는 3개(0·1·2), 평가는 시간 블록 5-fold다.
5. 사진 한 장 단위 점수와 함께 **최근 5장 다수결 판정**의 점수도 낸다.

**Drive `산공종설/`에 필요한 파일**
- `underfill_originals_4000.zip`: 원본 4,000장. 이미 올라가 있다.
- `pipeline_train_bundle_20261001.zip`: **새로 올릴 것**(작음). 학습 스크립트, 게이트 코드, 판정 파이프라인 코드, 분할표가 들어 있다.

노트북은 **파일 > 노트북 업로드**로 연다. 그다음 위에서부터 셀을 하나씩 실행한다(Shift+Enter).

## 1. GPU 확인
`assert`는 조건이 틀리면 그 자리에서 멈춘다. 모르고 CPU로 몇 시간 돌리는 걸 막아 준다.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아님: 런타임 > 런타임 유형 변경 > GPU"

## 2. Drive 연결 + 파일 확인
결과는 Drive(`results/pipeline_fc512_gate/`)에 바로 저장한다. 런타임이 끊기면 `/content`에 있던 파일은 사라지기 때문이다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
import os
DRIVE_DIR = "/content/drive/MyDrive/산공종설"
RESULTS = f"{DRIVE_DIR}/results/pipeline_fc512_gate"
os.makedirs(RESULTS, exist_ok=True)
for f in ("underfill_originals_4000.zip", "pipeline_train_bundle_20261001.zip"):
    print("OK  " if os.path.exists(f"{DRIVE_DIR}/{f}") else "없음", f)

## 3. 압축 해제
- `/content/orig`: 원본 사진. 마지막 줄에 4000이 나와야 정상이다.
- `/content/code`: 스크립트와 분할표(`folds.csv`)

In [ ]:
!cp "{DRIVE_DIR}/underfill_originals_4000.zip" "{DRIVE_DIR}/pipeline_train_bundle_20261001.zip" /content/
!unzip -q -o /content/underfill_originals_4000.zip -d /content/orig
!unzip -q -o /content/pipeline_train_bundle_20261001.zip -d /content/code
!ls /content/code
!find /content/orig -name "*.jpg" | wc -l

## 4. 2 epoch 점검 + 전체 시간 추정
ResNet18로 fold 0만 2 epoch 돌려서 에러가 없는지 본다.

- 로그에 `held 49 images {'normal': 19, 'underfill_50': 30}`이 나와야 한다. 게이트가 로컬에서 확인한 것과 똑같이 작동한다는 뜻이다.
- 처음에 사진 4,000장을 읽고 자르는 데 몇 분 걸린다. 본 실행에서도 모델·시드마다 한 번씩 이 시간이 든다.
- 전체 시간은 **2번째 epoch 시간**으로 추정한다. 1번째 epoch에는 GPU 준비 시간이 섞여서 길게 나온다.
- 나머지 두 모델은 노트북 GPU에서 잰 속도 비율(EffB0 약 1.5배, ConvNeXt 약 3.2배)로 어림한다. 대략적인 값이다.

In [ ]:
import csv
COMMON = ("--folds-csv /content/code/folds.csv --pilot-root /content/orig --input-column source "
          "--fixed-crop 0 250 1000 1250 --img-size 512 --modes finetune --fast --min-filament 0.25")
# 학습 끝에 나오는 긴 요약(JSON) 줄을 화면에서 숨기는 패턴. 같은 내용은 결과 폴더 CSV에 있다.
# 중괄호를 ! 줄에 직접 쓰면 Colab이 {COMMON} 같은 변수 치환을 통째로 건너뛰어서 변수로 뺐다.
HIDE = r"^ \|^{\|^}"
!rm -rf /content/smoke
!PYTHONUNBUFFERED=1 python /content/code/train_pilot_classifier.py {COMMON} --arch resnet18 --seeds 0 --folds 0 --epochs 2 --output /content/smoke | grep -v '{HIDE}'
sec = [float(r["epoch_seconds"]) for r in csv.DictReader(open("/content/smoke/runs/finetune/seed0/fold0/history.csv", encoding="utf-8-sig"))]
# 2번째 epoch 초 × 20 epoch × 5 fold × (ResNet18 1 + EffB0 1.5 + ConvNeXt 3.2) × 시드 3개
total_h = sec[1] * 20 * 5 * (1 + 1.5 + 3.2) * 3 / 3600
print(f"epoch별 초 {sec} -> 전체(모델 3 × 시드 3) 약 {total_h:.1f}시간 + 사진 읽기 9번")

## 5. 본 실행 (시드 → 모델 순으로 자동 반복)
**반복 순서는 시드 먼저다.** 시드 0으로 세 모델을 다 돌린 뒤 시드 1로 넘어간다. 도중에 끊겨도 세 모델을 한 번씩은 비교할 수 있다.

**끊겼을 때:** 이 셀을 그냥 다시 실행한다.
- 다 끝난 조합은 `summary.csv`가 있어서 건너뛴다.
- 돌다가 끊긴 조합은 결과가 반만 남아 있으므로 그 폴더를 지우고 처음부터 다시 돌린다. 코드가 알아서 한다.

자기 전에 이 셀과 6·7·8번 셀까지 한 번에 실행해 두면(런타임 > 이후 셀 실행), 다 끝난 뒤 결과를 정리하고 런타임을 반납한다.

In [ ]:
import os, shutil
ARCHS = ["resnet18", "efficientnet_b0", "convnext_tiny"]
SEEDS = [0, 1, 2]
for SEED in SEEDS:
    for ARCH in ARCHS:
        OUT = f"{RESULTS}/{ARCH}_seed{SEED}"
        if os.path.exists(f"{OUT}/summary.csv"):
            print("이미 끝남:", OUT)
            continue
        shutil.rmtree(OUT, ignore_errors=True)   # 반만 끝난 폴더 정리
        print(f"\n===== {ARCH} seed {SEED}")
        !PYTHONUNBUFFERED=1 python /content/code/train_pilot_classifier.py {COMMON} --arch {ARCH} --seeds {SEED} --output "{OUT}" | grep -v '{HIDE}'

## 6. 결과 모아 보기
끝난 조합만 모아 표로 만든다. 학습 도중에 실행해도 된다.

**표 읽는 법**
- `acc`, `underfill_recall`, `normal_recall`, `precision`, `auroc`: 사진 한 장 단위 점수다. 5개 fold의 검증 결과를 모아서 계산했다(게이트로 보류된 49장은 빠짐).
- `underfill_recall`: 실제 언더필 중 잡아낸 비율이다. `normal_recall`: 실제 정상을 정상으로 맞힌 비율이다.
- `fold_std_acc`: 시간 구간(fold)마다 정확도가 얼마나 흔들리는지다. 크면 출력 초반·후반 같은 특정 구간에 약하다는 뜻이다.
- `vote_…`: 최근 5장 중 3장 이상이 같은 판정일 때 내리는 **다수결 판정**의 점수다. 실제 현장에서 알람을 울릴 때 쓰는 방식이다. `decided`는 판정을 내린 비율이다. 처음 몇 장은 판정을 보류한다.
- 두 번째 표는 시드 3개를 평균한 값이다. `n`은 평균에 들어간 시드 수다.

**이 셀 출력을 복사해서 Claude에게 붙여 주면 된다.**

In [ ]:
import pandas as pd
rows = []
for ARCH in ARCHS:
    for SEED in SEEDS:
        OUT = f"{RESULTS}/{ARCH}_seed{SEED}"
        if not os.path.exists(f"{OUT}/summary.csv"):
            continue
        s = pd.read_csv(f"{OUT}/summary.csv", encoding="utf-8-sig").iloc[0]
        v = pd.read_csv(f"{OUT}/vote_metrics.csv", encoding="utf-8-sig").iloc[0]
        rows.append({"arch": ARCH, "seed": SEED, "acc": s.pooled_mean_accuracy,
                     "underfill_recall": s.pooled_mean_underfill_recall, "normal_recall": s.pooled_mean_normal_recall,
                     "precision": s.pooled_mean_underfill_precision, "auroc": s.pooled_mean_auroc,
                     "fold_std_acc": s.fold_std_accuracy, "decided": v.decided_rate, "vote_acc": v.vote_accuracy,
                     "vote_normal": v.vote_normal_recall, "vote_underfill": v.vote_underfill_recall,
                     "min_per_fold": s.mean_seconds_per_run / 60})
if not rows:
    print("아직 끝난 조합이 없음")
else:
    tab = pd.DataFrame(rows)
    print(tab.round(3).to_string(index=False))
    g = tab.drop(columns="seed").groupby("arch", sort=False)
    avg = g.mean().round(3)
    avg.insert(0, "n", g.size())
    print("\n[시드 평균]\n" + avg.to_string())

## 7. 결과 CSV 묶기
가중치(`.pt`)를 뺀 CSV·설정 파일만 zip으로 묶어 Drive `results/`에 저장한다. 크기가 작다. **이 zip을 내려받아서 Claude에게 주면** 오분류 사진 분석과 이전 실험 비교를 한다. 가중치는 Drive 각 폴더에 그대로 남는다.

In [ ]:
import zipfile, glob
ZIP = f"{DRIVE_DIR}/results/pipeline_fc512_gate_csv.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{RESULTS}/**/*", recursive=True):
        if p.endswith((".csv", ".json", ".txt")):
            z.write(p, os.path.relpath(p, RESULTS))
print(ZIP, f"{os.path.getsize(ZIP) / 2**20:.1f} MiB")

## 8. 런타임 반납
켜 둔 채로 두면 사용량이 계속 나간다. 이 셀은 Drive 저장을 마무리한 뒤 런타임을 끊는다.

In [ ]:
from google.colab import drive, runtime
drive.flush_and_unmount()
runtime.unassign()